# Experiment 10 — Physics-Grounded Passband & Acceptable-Region Selection

**Goal.** Treat frequency selection as a *graded* problem: one primary offset plus
clinically acceptable neighbours, each with a physical reason. The doctor noted that
neighbouring offsets can both be good.

**What is new vs Exp9**
1. `banded_fraction_myo` (already computed, previously unused) enters the composite.
2. Myocardial passband model (`PassbandEstimator`) — peak, acceptable set, rejected set, reasons.
3. Relative per-patient verdicts (no more absolute 0.60 cutoff that never fired).
4. LOOCV-learned fusion weights.
5. Post-hoc confidence calibration (isotonic / Platt).
6. Co-primary metrics: Top-1, clinical ±1 step, **acceptable-set hit**, MAE.

**Segmentation is used.** The consensus myocardium mask drives homogeneity, CNR,
`banded_fraction_myo`, `banding_burden`, and the passband peak.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from freqscout_agent.eval.exp10_runner import collect_patient_metrics, evaluate

print('Root:', ROOT)

## 1. Collect myocardial metrics (uses cached `exp8_seg_qc` masks)

In [ ]:
results = collect_patient_metrics()
print(f'Patients: {len(results)}')
# Quick mask QC
qc = pd.DataFrame([
    {'patient': pid, 'myo_px': r['myo_px'], 'n': len(r['offsets']),
     'gt_hz': r['gt_hz'], 'passband_peak_hz': r['offsets'][r['passband_peak']]}
    for pid, r in results.items()
])
print(qc.to_string(index=False))

## 2. Diagnostic curves

For each patient: `mu_myo`, `banded_fraction_myo`, homogeneity, CNR vs offset.
Vertical lines: GT (black) and passband peak (green).

In [ ]:
EXAMPLE = list(results.keys())[:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True)
axes = axes.ravel()
for ax, pid in zip(axes, EXAMPLE):
    r = results[pid]
    hz = r['offsets']
    ax.plot(hz, r['bfm'], 'r-o', label='banded_frac_myo', ms=4)
    ax.plot(hz, 1 - r['homog'], 'b--s', label='1-homog', ms=4)
    ax.axvline(r['gt_hz'], color='k', ls=':', label='GT')
    ax.axvline(r['offsets'][r['passband_peak']], color='g', ls='--', label='passband peak')
    ax.set_title(f"{pid}  MYO={r['myo_px']}px")
    ax.set_xlabel('Hz')
    ax.set_ylim(0, 1)
axes[0].legend(fontsize=7)
fig.suptitle('Intra-myocardial banding vs offset (lower is better)')
fig.tight_layout()
fig.savefig('exp10_passband_curves.png', dpi=120)
print('Saved exp10_passband_curves.png')

## 3. Does the unused physics signal beat the neural ranker?

Hypothesis: `argmin(banded_fraction_myo)` or the passband peak predicts GT
as well as, or better than, Exp5 neural-only.

In [ ]:
ev = evaluate(results)
print('=== Ablation ===')
print(pd.DataFrame(ev['ablation']).T.to_string())
print()
print('Default weights :', ev['weights_default'])
print('Grid weights    :', ev['weights_grid'])
print('LOOCV-deploy    :', ev['weights_loocv'])

## 4. Per-patient fused predictions + miss categories

In [ ]:
print(ev['df'][['patient_id','gt_hz','pred_hz','passband_peak','neural_pred','top1','within1','accept_hit','mae_hz']].to_string(index=False))
print()
print('=== Miss categorization ===')
if ev['misses']:
    print(pd.DataFrame(ev['misses'])[['patient_id','gt_hz','pred_hz','mae_hz','category']].to_string(index=False))
else:
    print('none')
ev['df'].to_csv('exp10_ablation_results.csv', index=False)

## 5. Confidence calibration (LOOCV)

Raw confidence = composite margin / 0.08.  Fit isotonic and Platt leave-one-out.

In [ ]:
iso, platt = ev['calibration_isotonic'], ev['calibration_platt']
print(f'Isotonic  ECE {iso.ece_before:.3f} → {iso.ece_after:.3f}')
print(f'          conf correct/wrong {iso.mean_conf_correct_after:.3f} / {iso.mean_conf_wrong_after:.3f}')
print(f'Platt     ECE {platt.ece_before:.3f} → {platt.ece_after:.3f}')
print(f'          conf correct/wrong {platt.mean_conf_correct_after:.3f} / {platt.mean_conf_wrong_after:.3f}')

## 6. Example reasoning (one patient)

Stage-A now emits a `reason` string per offset from the passband model.

In [ ]:
pid = 'Patient-BGT' if 'Patient-BGT' in results else list(results)[0]
r = results[pid]
print(pid, 'GT', r['gt_hz'], 'Hz | passband peak', r['offsets'][r['passband_peak']], 'Hz')
print('Acceptable set:', [r['offsets'][i] for i in r['acceptable']])
print('Rejected set  :', [r['offsets'][i] for i in r['rejected']])
print()
for i, reason in enumerate(r['reasons']):
    mark = ' *' if i == r['gt_slice'] else ''
    print(f"  {r['offsets'][i]:+7.0f} Hz  {reason}{mark}")

---
## Summary

Re-run `python run_exp10_eval.py` for a non-notebook dump of the same tables.
The agent pipeline (`run_exp9_pipeline.py` / `FreqScoutPipeline`) now uses the
same passband composite and graded acceptable/rejected output.